# Chapter 4. Sources and collection of data

*Companion notebook to* **Computational Criminology: Research Methods, Ethics, and Reproducible Practice with R and Python** *by Dr Fahad Hameed Khan.*

Run the setup cell once, then run the code cell. Everything uses synthetic data, so nothing here describes a real person. The R version of this chapter is in `code/r/ch04_fetching_api.R`.

In [ ]:
# Setup: fetch the repository, install what this chapter needs,
# and build the synthetic data. Safe to run more than once.
import os
if not os.path.exists('computational-criminology'):
    !git clone -q https://github.com/drfahadhameedkhan/computational-criminology.git
%cd -q computational-criminology
!pip install -q numpy pandas matplotlib requests
!python data/synthetic/make_synthetic_data.py

In [ ]:
from pathlib import Path
import pandas as pd

def fetch_live():
    # --- book code (Chapter 4) ---------------------------------------------
    # Python: one month of street-level crime from an open police API
    import requests
    url = "https://data.police.uk/api/crimes-street/all-crime"
    params = {"lat": 53.40, "lng": -2.97, "date": "2024-01"}
    r = requests.get(url, params=params, timeout=30)  # ask the service
    records = r.json()                                 # a list of records
    crimes = pd.json_normalize(records)                # flatten to a table
    return crimes

try:
    crimes = fetch_live()
    source = "live data.police.uk API"
except Exception as e:                                 # offline fallback
    DATA = Path("data/synthetic/incidents.csv")
    crimes = pd.read_csv(DATA)
    source = f"synthetic fallback ({type(e).__name__})"

print("source:", source)
print(crimes.shape)
cols = [c for c in ["category", "location.latitude"] if c in crimes.columns]
print(crimes[cols].head())

## Exercises

- For a research question of your choice, inventory three candidate data sources. For each, record the owner, the licence or terms of use, the documented coverage, and one important thing the documentation does not tell you.
- Coding task. Using the data.police.uk API or an equivalent open API in your jurisdiction, collect twelve months of street-level records for one force or city. Save the raw responses unaltered, then write a provenance README recording the endpoint, the dates of collection, the parameters used, and the licence.
- Take one open crime dataset and compare its published data dictionary against the file itself. List every field that is undocumented, every documented field that behaves differently from its description, and every code value that the dictionary does not explain.
- Draft the data section of a data-management plan for a project linking police records to another administrative source, addressing what will be collected, where it will be stored, who may access it, and when it will be destroyed.